# Phase 2A — saved-output diagnosis only

Selected C is fixed. No checkpoint loading, inference, training, Optuna, preprocessing, loss/sampler changes or model selection. Saved predictions are sufficient. Test distributions/errors are a frozen descriptive audit; proposed experiments below use training/validation evidence only.

Feature distributions use unique historical event rows per split to avoid weighting the same input event repeatedly; shared boundary context and temporal dependence still remain. Input/target correlations use one last-step feature vector per target (plus explicitly labeled window-mean summaries). Correlation is not feature importance or causation. Plot reuse avoids duplicate figures.


In [1]:
from pathlib import Path
import json,hashlib,time
import numpy as np
import pandas as pd
import torch
from scipy.stats import ks_2samp,wasserstein_distance,spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve()
if ROOT.name!="bilstm new updated":ROOT=ROOT/"bilstm new updated"
DATA=ROOT/"dataset"/"v2_timestamp_fixed";RESULTS=ROOT/"phase2_bilstm_results"
OUT=RESULTS/"error_diagnosis";OUT.mkdir(exist_ok=True)
def digest(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()
def serial(v):
    if isinstance(v,dict):return {str(k):serial(x) for k,x in v.items()}
    if isinstance(v,(list,tuple,np.ndarray)):return [serial(x) for x in v]
    if isinstance(v,(np.integer,)):return int(v)
    if isinstance(v,(np.bool_,)):return bool(v)
    if isinstance(v,(np.floating,float)):return float(v) if np.isfinite(v) else None
    return v
def save_json(name,obj):
    p=OUT/name;tmp=p.with_suffix(".tmp")
    tmp.write_text(json.dumps(serial(obj),indent=2,allow_nan=False),encoding="utf-8")
    for attempt in range(20):
        try:tmp.replace(p);break
        except PermissionError:
            if attempt==19:raise
            time.sleep(.1)
selection=json.loads((RESULTS/"selection_lock.json").read_text());assert selection["winner"]=="C"
original_registry=json.loads((RESULTS/"artifact_hashes.json").read_text())
source_paths=[RESULTS/name for name in original_registry["files"]]
source_paths += [RESULTS/"optuna_study.sqlite3",DATA/"bilstm_catalog_seq20.pt",
                DATA/"catalog_train_scalers.json",DATA/"split_manifest.csv"]
source_paths += [RESULTS/"variants"/v/name for v in "ABCD" for name in ["history.json","validation_predictions.npz","last.pt"]]
source_before={str(p.relative_to(ROOT)):digest(p) for p in source_paths}
for name,h in original_registry["files"].items():assert digest(RESULTS/name)==h
bundle=torch.load(DATA/"bilstm_catalog_seq20.pt",map_location="cpu",weights_only=True)
X={s:bundle["X_"+s].numpy() for s in ["train","val","test"]}
y={s:bundle["y_reg_"+s].numpy().astype(float) for s in X}
schema=json.loads((DATA/"feature_schema.json").read_text())
scaling=json.loads((DATA/"catalog_train_scalers.json").read_text())
names=[f["name"] for f in schema["ordered_features"]]
mean=np.array(scaling["scaler"]["mean"]);scale=np.array(scaling["scaler"]["scale"])
raw={s:X[s].astype(float)*scale+mean for s in X}
manifest=pd.read_csv(DATA/"split_manifest.csv",keep_default_na=False)
manifests={s:manifest.loc[manifest.split.eq(s)].reset_index(drop=True) for s in X}
histories={v:json.loads((RESULTS/"variants"/v/"history.json").read_text()) for v in "ABCD"}
comparison=pd.read_csv(RESULTS/"validation_comparison.csv")
val_pred={}
for v in "ABCD":
    with np.load(RESULTS/"variants"/v/"validation_predictions.npz",allow_pickle=False) as z:
        assert np.array_equal(z["actual"].astype(float),y["val"])
        val_pred[v]=z["prediction"].astype(float)
with np.load(RESULTS/"selected_test_predictions.npz",allow_pickle=False) as z:
    assert np.array_equal(z["actual"].astype(float),y["test"])
    assert str(z["selection_sha256"])==digest(RESULTS/"selection_lock.json")
    test_pred=z["prediction"].astype(float);recent_test=z["recent_event_baseline"].astype(float)
saved_metrics=json.loads((RESULTS/"selected_test_metrics.json").read_text())
def description(a):
    a=np.asarray(a,dtype=float)
    return {"n":len(a),"min":a.min(),"max":a.max(),"mean":a.mean(),"std":a.std(),
            "q05":np.quantile(a,.05),"q25":np.quantile(a,.25),"median":np.median(a),
            "q75":np.quantile(a,.75),"q95":np.quantile(a,.95)}
def errors(actual,pred):
    r=pred-actual
    return {"n":len(r),"mae":np.abs(r).mean(),"rmse":np.sqrt(np.mean(r*r)),"bias":r.mean(),
            "underprediction_count":int((r<0).sum()),"overprediction_count":int((r>0).sum()),
            "underprediction_rate":np.mean(r<0),"overprediction_rate":np.mean(r>0)}
def correlation(a,b,rank=False):
    if np.std(a)==0 or np.std(b)==0:return None
    return float(spearmanr(a,b).statistic if rank else np.corrcoef(a,b)[0,1])
print("Loaded v2 once and reused saved predictions; no checkpoint inference required.")


Loaded v2 once and reused saved predictions; no checkpoint inference required.


In [2]:
prediction_diagnostics={}
range_rows=[];error_frames=[]
bins=[(-np.inf,3.),(3.,4.),(4.,4.5),(4.5,5.),(5.,np.inf)]
bin_names=["<3","3–<4","4–<4.5","4.5–<5",">=5"]
for s,p in [("val",val_pred["C"]),("test",test_pred)]:
    actual=y[s];residual=p-actual
    prediction_diagnostics[s]={"actual":description(actual),"predicted":description(p),
      "errors":errors(actual,p),"prediction_to_actual_std_ratio":p.std()/actual.std(),
      "pearson_prediction_actual":correlation(actual,p),"spearman_prediction_actual":correlation(actual,p,True),
      "prediction_mean_minus_training_mean":p.mean()-y["train"].mean(),
      "prediction_mean_minus_training_median":p.mean()-np.median(y["train"]),
      "fraction_within_0_25_of_training_mean":np.mean(np.abs(p-y["train"].mean())<=.25),
      "absolute_error_quantiles":dict(zip(["q50","q90","q95","q99"],np.quantile(np.abs(residual),[.5,.9,.95,.99]))),
      "high":errors(actual[actual>=4.5],p[actual>=4.5]),
      "low_below3":errors(actual[actual<3],p[actual<3]),
      "huber_delta1_high_saturated_fraction":np.mean(np.abs(residual[actual>=4.5])>=1),
      "huber_delta1_all_saturated_fraction":np.mean(np.abs(residual)>=1)}
    frame=pd.DataFrame({"split":s,"target_event_id":manifests[s].target_event_id,
       "target_timestamp_utc":manifests[s].target_timestamp_utc,"actual":actual,"prediction":p,
       "residual":residual,"absolute_error":np.abs(residual),
       "last_historical_magnitude":raw[s][:,-1,0],
       "target_gap_days":manifests[s].input_to_target_gap_seconds.to_numpy()/86400.})
    error_frames.append(frame)
    for (lo,hi),label in zip(bins,bin_names):
        m=(actual>=lo)&(actual<hi)
        if m.any():range_rows.append({"split":s,"model":"BiLSTM_C","range":label,**errors(actual[m],p[m])})
error_analysis=pd.concat(error_frames,ignore_index=True)
error_analysis.to_csv(OUT/"error_analysis.csv",index=False)
outliers={s:frame.nlargest(10,"absolute_error").to_dict("records")
          for s,frame in error_analysis.groupby("split",sort=False)}
assert np.isclose(prediction_diagnostics["test"]["errors"]["mae"],saved_metrics["metrics"]["mae"],atol=1e-12)
# Existing constants are reconstructed exactly from training targets, not fitted anew.
baseline_metrics={};baseline_rows=[]
for s,recent in [("val",raw["val"][:,-1,0]),("test",recent_test)]:
    predictors={"training_mean":np.full(len(y[s]),y["train"].mean()),
                "training_median":np.full(len(y[s]),np.median(y["train"])),
                "recent_event_magnitude":recent}
    baseline_metrics[s]={}
    for name,p in predictors.items():
        high=y[s]>=4.5
        baseline_metrics[s][name]={"overall":errors(y[s],p),"high":errors(y[s][high],p[high])}
        for (lo,hi),label in zip(bins,bin_names):
            m=(y[s]>=lo)&(y[s]<hi)
            if m.any():baseline_rows.append({"split":s,"model":name,"range":label,**errors(y[s][m],p[m])})
for saved in saved_metrics["baselines"]:
    assert np.isclose(baseline_metrics["test"][saved["model"]]["overall"]["mae"],saved["mae"],atol=1e-12)
save_json("magnitude_range_errors.json",range_rows+baseline_rows)
print("Saved magnitude-range errors and per-event residuals.")


Saved magnitude-range errors and per-event residuals.


## Distribution and relationship audit

Training-versus-validation comparisons are the development evidence. Test comparisons below are explicitly descriptive and must not select fixes. KS distances and Wasserstein distances are effect-size summaries; no IID significance claim is made. Standardized mean differences use training standard deviation; Wasserstein normalization uses training IQR. Feature correlations include period-specific train halves and validation, rather than interpreting an aggregate temporal trend as predictive information.


In [3]:
unique_features={};unique_ids={}
for s in X:
    ids=np.array([json.loads(v) for v in manifests[s].input_event_ids]).ravel()
    ids_unique,first=np.unique(ids,return_index=True)
    unique_ids[s]=set(ids_unique);unique_features[s]=raw[s].reshape(-1,10)[first]
feature_distributions={s:{name:description(unique_features[s][:,j]) for j,name in enumerate(names)} for s in X}
target_distributions={s:{**description(y[s]),"high_count":int((y[s]>=4.5).sum()),
 "high_fraction":np.mean(y[s]>=4.5),"target_start":manifests[s].target_timestamp_utc.iloc[0],
 "target_end":manifests[s].target_timestamp_utc.iloc[-1],
 "target_gap_days":description(manifests[s].input_to_target_gap_seconds.to_numpy()/86400.)} for s in X}
def distribution_difference(a,b):
    denom=np.std(a)
    iqr=np.quantile(a,.75)-np.quantile(a,.25)
    return {"mean_difference":np.mean(b)-np.mean(a),"standardized_mean_difference":(np.mean(b)-np.mean(a))/denom if denom else None,
            "KS_distance":float(ks_2samp(a,b,method="asymp").statistic),
            "wasserstein_original_units":wasserstein_distance(a,b),
            "wasserstein_over_training_IQR":wasserstein_distance(a,b)/iqr if iqr else None}
shift={s:{"target":distribution_difference(y["train"],y[s]),
           "features":{name:distribution_difference(unique_features["train"][:,j],unique_features[s][:,j])
                       for j,name in enumerate(names)},
           "target_gap_days":distribution_difference(manifests["train"].input_to_target_gap_seconds.to_numpy()/86400.,
                                                     manifests[s].input_to_target_gap_seconds.to_numpy()/86400.)}
       for s in ["val","test"]}
min_train=X["train"].min(axis=(0,1));max_train=X["train"].max(axis=(0,1))
range_coverage={}
for s in X:
    outside=(X[s]<min_train)|(X[s]>max_train)
    bad=outside.any(axis=(1,2))
    p=val_pred["C"] if s=="val" else test_pred if s=="test" else None
    range_coverage[s]={"out_of_range_windows":int(bad.sum()),"fraction":bad.mean(),
      "feature_window_fractions":dict(zip(names,outside.any(axis=1).mean(axis=0)))}
    if p is not None:
        range_coverage[s]["out_of_range_MAE"]=float(np.abs(p[bad]-y[s][bad]).mean()) if bad.any() else None
        range_coverage[s]["in_range_MAE"]=float(np.abs(p[~bad]-y[s][~bad]).mean()) if (~bad).any() else None
relationship_rows=[]
for s in X:
    for summary_name,features in [("last_historical_step",raw[s][:,-1,:]),("mean_over_20_steps",raw[s].mean(axis=1))]:
        for j,name in enumerate(names):
            relationship_rows.append({"split":s,"summary":summary_name,"feature":name,
               "pearson":correlation(features[:,j],y[s]),"spearman":correlation(features[:,j],y[s],True)})
for period,index in [("train_first_half",np.arange(len(y["train"])//2)),
                     ("train_second_half",np.arange(len(y["train"])//2,len(y["train"])))]:
    for j,name in enumerate(names):
        relationship_rows.append({"split":period,"summary":"last_historical_step","feature":name,
          "pearson":correlation(raw["train"][index,-1,j],y["train"][index]),
          "spearman":correlation(raw["train"][index,-1,j],y["train"][index],True)})
relationships=pd.DataFrame(relationship_rows)
relationships.to_csv(OUT/"feature_target_relationships.csv",index=False)
redundancy=pd.DataFrame(unique_features["train"],columns=names).corr(method="spearman")
redundant_pairs=[{"feature1":names[i],"feature2":names[j],"training_spearman":redundancy.iloc[i,j]}
                for i in range(10) for j in range(i+1,10) if abs(redundancy.iloc[i,j])>=.8]
# Frozen train-defined quintile bins: a descriptive relationship check, not a fitted predictor.
conditional=[]
for j,name in enumerate(names):
    interior=np.unique(np.quantile(raw["train"][:,-1,j],[.2,.4,.6,.8]))
    edges=np.r_[-np.inf,interior,np.inf]
    for s in ["train","val"]:
        groups=np.searchsorted(interior,raw[s][:,-1,j],side="right")
        for b in range(len(edges)-1):
            m=groups==b
            if m.any():conditional.append({"feature":name,"split":s,"train_defined_bin":b,"n":int(m.sum()),
                "mean_target":float(y[s][m].mean()),"high_fraction":float((y[s][m]>=4.5).mean())})
save_json("feature_distribution_diagnostics.json",{"unique_historical_event_counts":{s:len(a) for s,a in unique_features.items()},
 "feature_distributions":feature_distributions,"shifts":shift,"range_coverage":range_coverage,
 "training_redundant_pairs_abs_spearman_ge_0_8":redundant_pairs,"train_defined_conditional_summaries":conditional,
 "shared_context_counts":{"train_val":len(unique_ids["train"]&unique_ids["val"]),"val_test":len(unique_ids["val"]&unique_ids["test"])}})
print("Computed split shifts and descriptive feature relationships; no feature importance inferred.")


Computed split shifts and descriptive feature relationships; no feature importance inferred.


In [4]:
overfitting={};sampling={}
for row in comparison.itertuples(index=False):
    v=row.variant;history=histories[v];best=history[row.best_epoch-1];last=history[-1]
    flagged=[h["epoch"] for h in history if h["epoch"]>row.best_epoch and
             h["val_mae"]>best["val_mae"]+.01 and h["train_mae"]<best["train_mae"]]
    persistent=[e for e in flagged if e+1 in flagged and e+2 in flagged]
    overfitting[v]={"best_epoch":int(row.best_epoch),"epochs_run":len(history),
      "best_clean_train_MAE":best["train_mae"],"best_validation_MAE":best["val_mae"],
      "MAE_gap_at_best":best["val_mae"]-best["train_mae"],
      "Huber_gap_at_best":best["val_huber"]-best["train_huber"],
      "post_best_validation_MAE_change":last["val_mae"]-best["val_mae"],
      "post_best_train_MAE_change":last["train_mae"]-best["train_mae"],
      "late_pattern_flagged_epochs":flagged,"first_three_epoch_pattern":persistent[0] if persistent else None,
      "stopped_after_8_nonimproving_epochs":len(history)-row.best_epoch==8}
    sampling[v]={"draws_per_epoch":4973,"epochs":len(history),
      "unique_parents_min":min(h["unique_parents"] for h in history),
      "unique_parents_max":max(h["unique_parents"] for h in history),
      "unique_parents_mean":np.mean([h["unique_parents"] for h in history]),
      "repeated_draws_mean":np.mean([h["repeated_draws"] for h in history]),
      "repeated_draws_total":sum(h["repeated_draws"] for h in history),
      "effective_high_fraction_mean":np.mean([h["effective_high_fraction"] for h in history]),
      "same_window_across_epochs":"Expected in all variants; repeat_draws here counts extra appearances within an epoch."}
with np.load(ROOT/"augmentation_experiment/v2_timestamp_fixed/sampling_plan.npz",allow_pickle=False) as z:
    weights=z["weights"].copy();epoch0=z["parent_rows_C"].copy()
exposure=np.bincount(epoch0,minlength=len(y["train"]))
sampling["weighted_design"]={"weights_min":weights.min(),"weights_max":weights.max(),
 "Kish_effective_weight_count":weights.sum()**2/np.sum(weights**2),
 "expected_high_probability":weights[y["train"]>=4.5].sum()/weights.sum(),
 "epoch0_unique":int((exposure>0).sum()),"epoch0_repeated":int(len(epoch0)-(exposure>0).sum()),
 "epoch0_max_draws_one_parent":int(exposure.max()),
 "epoch0_parent_exposure_histogram":{str(k):int((exposure==k).sum()) for k in np.unique(exposure)},
 "cumulative_distinct_parents":"Not recoverable from hash-only epoch histories; not inferred.",
 "interpretation":"Finite repeats increase reuse risk, but no memorization measurement or causal attribution is available."}
paired_comparisons={}
for a,b in [("A","B"),("C","D"),("A","C")]:
    loss_delta=np.abs(val_pred[a]-y["val"])-np.abs(val_pred[b]-y["val"])
    intervals={}
    for block in [20,50]:
        rng=np.random.default_rng(42)
        starts=rng.integers(0,len(loss_delta),size=(2000,int(np.ceil(len(loss_delta)/block))))
        idx=((starts[...,None]+np.arange(block))%len(loss_delta)).reshape(2000,-1)[:,:len(loss_delta)]
        means=loss_delta[idx].mean(axis=1)
        intervals[str(block)]=np.quantile(means,[.025,.975]).tolist()
    paired_comparisons[a+"_minus_"+b]={"mean_MAE_improvement_for_second":loss_delta.mean(),
       "circular_block_bootstrap_95pct_interval":intervals,
       "caution":"Descriptive paired-loss uncertainty only; arbitrary block-length sensitivity, single trained seed and validation selection not accounted for."}
loss_diagnosis={
 "implemented_loss":"Huber delta=1; dL/dprediction=clip(prediction-target,-1,1)",
 "validation_high_saturated_fraction":prediction_diagnostics["val"]["huber_delta1_high_saturated_fraction"],
 "validation_all_saturated_fraction":prediction_diagnostics["val"]["huber_delta1_all_saturated_fraction"],
 "observed_weighted_validation_high_MAE_improvement_A_to_C":
    float(comparison.set_index("variant").loc["A","validation_high_mae"]-comparison.set_index("variant").loc["C","validation_high_mae"]),
 "confirmed":"Symmetric Huber bounds prediction-space influence for large residuals; measured high-residual saturation is descriptive.",
 "hypothesis":"A robust average loss with limited stable predictive information can favor common magnitudes; neither Huber nor imbalance alone is proven causal.",
 "no_alternative_loss_evaluated":True,
 "source":"https://docs.pytorch.org/docs/2.15/generated/torch.nn.HuberLoss.html"}
print("History and sampling counts audited; paired validation effects calculated without new model fits.")


History and sampling counts audited; paired validation effects calculated without new model fits.


In [5]:
# Reuse existing actual/predicted, magnitude-error and A/B/C/D curves; create only missing requested views.
reused_figures={
 "actual_vs_predicted":"../plots/selected_test_predictions_errors.png",
 "error_by_magnitude":"../plots/test_error_by_magnitude.png",
 "training_validation_MAE":"../plots/training_validation_mae.png",
 "training_validation_Huber":"../plots/training_validation_huber.png"}
assert all((OUT/path).exists() for path in reused_figures.values())
fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,s,p in [(axes[0],"val",val_pred["C"]),(axes[1],"test",test_pred)]:
    ax.scatter(y[s],p-y[s],s=10,alpha=.4)
    ax.axhline(0,color="black",ls="--");ax.axvline(4.5,color="gray",ls=":")
    ax.set(xlabel="Actual magnitude (ML)",ylabel="Prediction minus actual (ML)",title=s+" — selected C")
fig.tight_layout();fig.savefig(OUT/"residuals_vs_actual.png",dpi=150);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4))
for s in y:ax.hist(y[s],bins=np.arange(1.4,9.01,.2),density=True,histtype="step",linewidth=1.8,label=s)
ax.axvline(4.5,color="gray",ls=":");ax.set(xlabel="Next-event magnitude (ML)",ylabel="Density",title="Frozen target distributions")
ax.legend();fig.tight_layout();fig.savefig(OUT/"target_distribution_by_split.png",dpi=150);plt.close(fig)
metrics={"selected_variant":"C","source_run_id":selection["run_id"],"prediction_errors":prediction_diagnostics,
 "target_distributions":target_distributions,"distribution_shift":shift,"feature_distributions":feature_distributions,
 "feature_range_coverage":range_coverage,"feature_target_correlations":relationship_rows,
 "feature_redundancy_training":redundant_pairs,"overfitting":overfitting,"sampling":sampling,
 "paired_validation_comparisons":paired_comparisons,"loss_diagnosis":loss_diagnosis,
 "baseline_review":baseline_metrics,"magnitude_ranges":range_rows+baseline_rows,"extreme_error_examples":outliers,
 "reused_figures":reused_figures,"frozen_test_audit_only":True,"inference_runs":0,"training_runs":0,"optuna_runs":0,
 "recommendation_evidence_scope":"Training/validation only; no test-driven configuration changes.",
 "limitations":["Correlations do not establish feature importance, causality or impossibility of prediction.",
 "Shared/overlapping context and chronology invalidate IID interpretations.",
 "One training seed and reused validation selection limit robustness claims.",
 "Operational reporting availability, magnitude calibration and completeness remain unknown."]}
save_json("diagnostic_metrics.json",metrics)
print(json.dumps({"prediction_mean":prediction_diagnostics["test"]["predicted"]["mean"],
 "prediction_std":prediction_diagnostics["test"]["predicted"]["std"],
 "actual_std":prediction_diagnostics["test"]["actual"]["std"],
 "train_mean":y["train"].mean(),
 "target_mean_by_split":{s:float(y[s].mean()) for s in y},
 "out_of_range_fraction":{s:float(range_coverage[s]["fraction"]) for s in y}},indent=2))


{
  "prediction_mean": 3.2717629788406786,
  "prediction_std": 0.07786868726403522,
  "actual_std": 0.6414080836921238,
  "train_mean": 4.033842747057561,
  "target_mean_by_split": {
    "train": 4.033842747057561,
    "val": 3.510610325907318,
    "test": 3.1012183733933867
  },
  "out_of_range_fraction": {
    "train": 0.0,
    "val": 0.46854460093896716,
    "test": 1.0
  }
}


In [6]:
val=prediction_diagnostics["val"];test=prediction_diagnostics["test"]
val_rel=relationships[(relationships.split=="val")&(relationships.summary=="last_historical_step")]
train_rel=relationships[(relationships.split=="train")&(relationships.summary=="last_historical_step")]
top_val=val_rel.loc[val_rel.spearman.abs().idxmax()]
top_train=train_rel.loc[train_rel.spearman.abs().idxmax()]
top_shifts=sorted(shift["val"]["features"].items(),key=lambda pair:abs(pair[1]["standardized_mean_difference"] or 0),reverse=True)[:3]
recommendations=[
 {"rank":1,"category":"Loss function adjustment",
  "evidence":f"Validation C underpredicts all {val['high']['n']} high targets; {100*val['huber_delta1_high_saturated_fraction']:.1f}% of their residuals exceed Huber delta 1.",
  "expected_benefit":"Test whether increasing large-error influence reduces upper-tail shrinkage.",
  "tradeoff":"May raise overall MAE/low-magnitude overprediction and amplify questionable extreme labels.",
  "validation_only_experiment":"FIRST: fixed C parents, architecture, AdamW, seed and stopping; compare existing Huber delta=1 with delta=2 only. Report overall/high MAE, bias and prediction spread. No Optuna or test access; confirm across seeds before claiming a gain."},
 {"rank":2,"category":"Feature engineering",
  "evidence":f"Strongest last-step |Spearman|: train {abs(top_train.spearman):.3f}, validation {abs(top_val.spearman):.3f}; correlations vary across periods.",
  "expected_benefit":"Determine whether current historical summaries provide stable next-event information before adding complexity.",
  "tradeoff":"New regional/context indicators can overfit or require unavailable reporting information.",
  "validation_only_experiment":"Fit a small train-only regularized linear probe on existing last-step/window-mean features in a future task; compare recent-event baseline on validation, then test one justified causal feature at a time."},
 {"rank":3,"category":"Magnitude-aware weighting",
  "evidence":f"C improves validation high MAE over A by {loss_diagnosis['observed_weighted_validation_high_MAE_improvement_A_to_C']:.4f}, yet high underprediction stays 100%.",
  "expected_benefit":"Express upper-tail error cost directly instead of assuming sampling alone is sufficient.",
  "tradeoff":"Worse overall calibration/MAE and disproportionate influence of uncertain high ML values.",
  "validation_only_experiment":"After isolating the loss, keep the sampler fixed and compare no added loss weights versus a bounded high-target weight fixed from training only; evaluate the validation tradeoff."},
 {"rank":4,"category":"Sampling strategy",
  "evidence":f"C averages {sampling['C']['unique_parents_mean']:.0f} unique parents and {sampling['C']['repeated_draws_mean']:.0f} repeat draws per {4973}-draw epoch.",
  "expected_benefit":"Retain high-target exposure with less duplicate concentration.",
  "tradeoff":"Capping repeats changes effective target exposure and may reduce tail gains.",
  "validation_only_experiment":"Compare approved weighted draws with a conservative per-parent repeat cap, keeping epoch size fixed; record achieved exposure and validation results without changing loss simultaneously."},
 {"rank":5,"category":"Regularization",
  "evidence":"C/D show post-best train improvement while validation worsens; selected best checkpoints already limit this.",
  "expected_benefit":"Reduce late memorization if it is material.",
  "tradeoff":"May further compress predictions or worsen underfitting; early stopping already helps.",
  "validation_only_experiment":"One fixed higher-dropout setting against current C, same loss/sampling; compare best-epoch validation and prediction spread."},
 {"rank":6,"category":"Alternative architecture",
  "evidence":"Current results show little stable explained validation variance; no evidence identifies BiLSTM structure as the cause.",
  "expected_benefit":"Check whether a simpler sequence model generalizes as well with fewer parameters.",
  "tradeoff":"Architecture search can consume validation information without fixing weak inputs or calibration.",
  "validation_only_experiment":"One smaller unidirectional recurrent baseline under the same split/loss/sampling protocol, only after simpler ablations; no CNN/fusion."}
]
save_json("ranked_next_experiments.json",recommendations)
metrics["ranked_next_experiments"]=recommendations
range_table=[]
for row in range_rows:
    if row["split"]=="test":
        range_table.append(f"| {row['range']} | {row['n']} | {row['mae']:.4f} | {row['bias']:+.4f} | {100*row['underprediction_rate']:.1f}% |")
split_table=[f"| {s} | {len(y[s])} | {y[s].mean():.3f} | {100*(y[s]>=4.5).mean():.2f}% | {target_distributions[s]['target_gap_days']['median']:.3f} |" for s in y]
overfit_table=[f"| {v} | {overfitting[v]['best_epoch']}/{overfitting[v]['epochs_run']} | {overfitting[v]['MAE_gap_at_best']:+.4f} | {overfitting[v]['post_best_validation_MAE_change']:+.4f} | {overfitting[v]['post_best_train_MAE_change']:+.4f} | {sampling[v]['unique_parents_mean']:.0f}/{sampling[v]['repeated_draws_mean']:.0f} |" for v in "ABCD"]
recommendation_text="\n".join(f"{r['rank']}. **{r['category']}** — Evidence: {r['evidence']} Expected benefit: {r['expected_benefit']} Tradeoff: {r['tradeoff']} Next test: {r['validation_only_experiment']}" for r in recommendations)
baseline_table=[f"| {name} | {values['overall']['mae']:.4f} | {values['overall']['bias']:+.4f} | {values['high']['mae']:.4f} | {100*values['high']['underprediction_rate']:.1f}% |" for name,values in baseline_metrics["test"].items()]
report=f"""# Phase 2A: BiLSTM error diagnosis
**No training, inference, checkpoint loading or experiment changes.** Selected C and its test results remain frozen.

## Confirmed prediction behavior
High-magnitude underprediction is systematic: validation **{val['high']['underprediction_count']}/{val['high']['n']}**, test **{test['high']['underprediction_count']}/{test['high']['n']}**. Test high MAE/bias are {test['high']['mae']:.4f}/{test['high']['bias']:+.4f}.
For test ML<3, **{test['low_below3']['overprediction_count']}/{test['low_below3']['n']}** are overpredicted; mean bias {test['low_below3']['bias']:+.4f}.

Test predictions span {test['predicted']['min']:.3f}–{test['predicted']['max']:.3f}, mean {test['predicted']['mean']:.3f}, SD {test['predicted']['std']:.4f}; actual SD is {test['actual']['std']:.4f}. Prediction SD is only {100*test['prediction_to_actual_std_ratio']:.1f}% of actual SD. Prediction/actual Pearson correlation is {test['pearson_prediction_actual']:.3f}. This confirms strong compression toward common magnitudes. The global training mean is {y['train'].mean():.3f}; predictions center {test['prediction_mean_minus_training_mean']:+.3f} away from it, so literal collapse to that mean is unsupported.
Validation already shows prediction SD {val['predicted']['std']:.4f} versus actual SD {val['actual']['std']:.4f}.

| Test ML range | Count | MAE | Signed bias | Underpredicted |
|---|---:|---:|---:|---:|
{chr(10).join(range_table)}

Largest test error: {outliers['test'][0]['target_event_id']}, actual {outliers['test'][0]['actual']:.2f}, prediction {outliers['test'][0]['prediction']:.3f}, absolute error {outliers['test'][0]['absolute_error']:.3f}. Top-ten examples with IDs/times are in diagnostic_metrics.json; all validation/test residuals are in error_analysis.csv. These are errors, not evidence that source events are invalid.

## Chronological distribution shift
| Split | Targets | Mean ML | ML>=4.5 | Median target gap (days) |
|---|---:|---:|---:|---:|
{chr(10).join(split_table)}

Train→validation target KS distance {shift['val']['target']['KS_distance']:.3f}, Wasserstein {shift['val']['target']['wasserstein_original_units']:.3f} ML; train→test {shift['test']['target']['KS_distance']:.3f}/{shift['test']['target']['wasserstein_original_units']:.3f} ML. These are substantial descriptive population changes, not proof of a causal explanation for model error.

Largest absolute train→validation feature mean shifts, in training SD units: {', '.join(f"{name} {values['standardized_mean_difference']:+.2f}" for name,values in top_shifts)}.
Input latitude/longitude means (train→validation): {feature_distributions['train']['historical_latitude_deg']['mean']:.2f}/{feature_distributions['train']['historical_longitude_deg']['mean']:.2f} → {feature_distributions['val']['historical_latitude_deg']['mean']:.2f}/{feature_distributions['val']['historical_longitude_deg']['mean']:.2f}.
Mean 30-day rate: {feature_distributions['train']['event_rate_30d']['mean']:.3f} → {feature_distributions['val']['event_rate_30d']['mean']:.3f} events/day.
Outside-training-range windows: validation {100*range_coverage['val']['fraction']:.2f}%, test {100*range_coverage['test']['fraction']:.2f}%. All ten features, tails, gap distributions and in/out-of-range errors are saved. Shift can occur inside training ranges, so range coverage alone is insufficient.

Feature comparisons deduplicate repeated historical event rows within each split. Boundary context still overlaps and all series are temporally dependent; no IID significance tests are claimed. Test comparisons are frozen description and are not used to rank interventions.

## Overfitting and sampling
| Variant | Best/run epoch | MAE gap at best | Later val MAE change | Later train MAE change | Mean unique/repeat draws per epoch |
|---|---:|---:|---:|---:|---:|
{chr(10).join(overfit_table)}

C/D select epoch 12 versus A/B 14, and show late validation degradation with training improvement. This is consistent with some late overfitting, not proof that weighted sampling caused memorization or that best-epoch error is primarily overfitting. All runs stopped eight nonimproving epochs after their best and restored that checkpoint.

Weighted C/D increase high-target exposure from {100*(y['train']>=4.5).mean():.2f}% to an epoch-average {100*sampling['C']['effective_high_fraction_mean']:.2f}%. Epoch zero has {sampling['weighted_design']['epoch0_unique']} unique windows, {sampling['weighted_design']['epoch0_repeated']} repeated draws, and maximum {sampling['weighted_design']['epoch0_max_draws_one_parent']} appearances of one parent. Weights span {weights.min():.2f}–{weights.max():.2f}. Exposure changes are real; repetitions add no independent information. High-target coverage alone does not explain failure: all four variants underpredict all validation high targets.

Noise A→B validation MAE improvement is {paired_comparisons['A_minus_B']['mean_MAE_improvement_for_second']:+.6f}; C→D is {paired_comparisons['C_minus_D']['mean_MAE_improvement_for_second']:+.6f}. Twenty-step circular-block intervals: {paired_comparisons['A_minus_B']['circular_block_bootstrap_95pct_interval']['20']} and {paired_comparisons['C_minus_D']['circular_block_bootstrap_95pct_interval']['20']}. Fifty-step sensitivity is saved. These tiny single-seed effects do not establish a reliable noise benefit; bootstrap intervals omit training/selection uncertainty.

## Loss and feature informativeness
[Huber loss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.HuberLoss.html) is symmetric and limits prediction-space gradient magnitude beyond delta. With delta=1, {100*val['huber_delta1_high_saturated_fraction']:.1f}% of validation high-target errors are in that region. Reduced outlier influence could contribute to conservative predictions, but no loss ablation exists; Huber does not inherently require downward bias.

Available inputs are past magnitude/depth/location, elapsed time, rate and rolling magnitude/depth summaries. They contain historical context, with no measured precursory or target-event information. Strongest last-step target |Spearman| is {abs(top_train.spearman):.3f} in training ({top_train.feature}) and {abs(top_val.spearman):.3f} in validation ({top_val.feature}). Full Pearson/Spearman, window-mean summaries, training halves and training-defined conditional bins are saved. Training feature pairs with |Spearman|>=0.8: {len(redundant_pairs)}.
Period changes and limited stable associations support investigating feature sufficiency; they do not establish causal importance, prove nonlinear signal is absent, or prove the task inherently unpredictable.

## Baseline review
| Test baseline | Overall MAE | Overall bias | High ML MAE | High underprediction |
|---|---:|---:|---:|---:|
{chr(10).join(baseline_table)}

C beats these baselines overall (MAE {test['errors']['mae']:.4f}), but training constants can have smaller upper-tail errors by predicting larger values everywhere. This is a tradeoff, not superior event discrimination. Constants were reconstructed from training values and checked against saved baseline metrics; recent-event test predictions were reused. No new model was fitted.

## Most plausible explanation and next experiments
**Hypothesis:** weak or nonstationary input–target information, combined with an average robust objective, produces a narrow common-magnitude prediction band; chronological distribution shift worsens transfer. Mild late overfitting and limited tail emphasis may contribute. Their causal contributions are not identified by this audit. Class imbalance alone is insufficient as an explanation.

Ranked below using **training/validation evidence only**; no proposals were implemented:
{recommendation_text}

## Scope, figures and preservation
Reuse [actual/predicted](../plots/selected_test_predictions_errors.png), [magnitude errors](../plots/test_error_by_magnitude.png), [MAE curves](../plots/training_validation_mae.png) and [Huber curves](../plots/training_validation_huber.png).
New: [residuals vs actual](residuals_vs_actual.png), [split targets](target_distribution_by_split.png).
Notebook: ../../phase2a_bilstm_error_diagnosis.ipynb. Original datasets, checkpoints, Optuna database, predictions, metrics and selection are unchanged. Reporting availability, magnitude calibration and catalog completeness remain UNKNOWN. This audit does not establish operational EEW.
"""
(OUT/"diagnosis_report.md").write_text(report,encoding="utf-8")
assert all(digest(ROOT/path)==h for path,h in source_before.items())
metrics["preservation_verification"]={"status":"PASS","source_sha256":source_before,
 "checkpoints_loaded":0,"model_inference_calls":0,"fits":0,"optimization_runs":0}
save_json("diagnostic_metrics.json",metrics)
for path in [ROOT/"project_status.md",ROOT/".agent-memory/checkpoint.md"]:
    start="<!-- PHASE2A_CURRENT_START -->";end="<!-- PHASE2A_CURRENT_END -->"
    old=path.read_text(encoding="utf-8")
    if start in old and end in old:old=old.split(end,1)[1].lstrip()
    note=f"""# Phase 2A diagnosis complete (2026-11-09)
- Saved-output audit only; no checkpoint inference/training/tuning or source changes. Report: phase2_bilstm_results/error_diagnosis/diagnosis_report.md.
- C predictions concentrate at mean {test['predicted']['mean']:.3f}, SD {test['predicted']['std']:.4f} vs actual {test['actual']['std']:.4f}; training mean {y['train'].mean():.3f}. High underprediction {test['high']['underprediction_count']}/{test['high']['n']}; low (<3) overprediction {test['low_below3']['overprediction_count']}/{test['low_below3']['n']}.
- Target means train/val/test: {y['train'].mean():.3f}/{y['val'].mean():.3f}/{y['test'].mean():.3f}. Feature/relationship shifts and late C/D overfitting documented; causal contributions remain hypotheses.
- First proposed future ablation: fixed-C Huber delta 1 vs 2 using validation only. No intervention implemented; test remains frozen. Ranked alternatives and tradeoffs saved.
- STOP after diagnosis; wait for user instruction. Operational availability/calibration remain UNKNOWN.

## Earlier notes
"""
    path.write_text(start+"\n"+note+end+"\n\n"+old,encoding="utf-8")
print("Diagnostic report saved; originals verified unchanged. No model or data modifications.")


Diagnostic report saved; originals verified unchanged. No model or data modifications.
